# Remember · Colab T4

수정본을 `/content/GRT`에 pull한 뒤 실행합니다. 기본은 RMT, `MODEL='grt'`로 동일 경로의 GRT를 선택합니다.
D128 / 4층 / memory32 / FP32, train100만 / validation1천 / test1만.
B2048, 1쌍 최대500 → 2쌍 최대2500 update, 고정 길이 생성 정확도99%에서 단계 전환합니다.
진행 중인 이전 버전 실험은 기존 checkout에서 마치고 이 노트북에는 새 실행 경로를 사용하세요.
GRT의 T4 처리량·VRAM·수렴은 후속 실험에서 확인합니다. W&B는 기본 비활성입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile
import torch, yaml

REPO_DIR = Path('/content/GRT')
MODEL = 'rmt'  # 'grt'도 동일한 데이터·학습·평가 경로 사용
WANDB = False
CONFIGS = ['configs/base.yaml', f'configs/{MODEL}.yaml', 'configs/t4.yaml']
RUN_DIR = Path(f'/content/drive/MyDrive/GRT/runs/{MODEL}_remember_v2_t4_b2048_seed54')
DEVICE = 'cuda'


In [ ]:
assert MODEL in ('rmt', 'grt')
assert (REPO_DIR / 'configs/base.yaml').exists(), '수정본 checkout을 사용하세요.'
if DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'GPU runtime이 필요합니다.'
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR) + ('[wandb]' if WANDB else '')],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
if installation.returncode:
    print(installation.stdout)
    installation.check_returncode()
sys.path.insert(0, str(REPO_DIR / 'src'))
from grt.config import load_config, validate_config
# W&B 키는 YAML에 저장하지 않습니다. 활성화 시 로그인 환경을 사용합니다.
if WANDB:
    import wandb
    wandb.login()
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as override:
    yaml.safe_dump({'wandb': {'enabled': WANDB}}, override)
    OVERRIDE = override.name
CONFIG_PATHS = [str(REPO_DIR / path) for path in CONFIGS] + [OVERRIDE]
cfg = load_config(CONFIG_PATHS, output_dir=RUN_DIR)
validate_config(cfg)
print('Model:', MODEL, '| effective batch:', cfg.training.batch_size * cfg.training.grad_accum_steps)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Run:', RUN_DIR)


In [ ]:
# 동일 설정·경로로 다시 실행하면 last.pt에서 재개합니다.
subprocess.run([
    sys.executable, '-u', 'scripts/train.py', '--config', *CONFIG_PATHS,
    '--output-dir', str(RUN_DIR), '--device', DEVICE,
], cwd=REPO_DIR, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.loads((RUN_DIR / 'summary.json').read_text())
print((RUN_DIR / 'summary.txt').read_text())
print((RUN_DIR / 'evaluation.txt').read_text())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
offset = 0
for index, stage in enumerate(summary['stages'], 1):
    stage_dir = RUN_DIR / f"stage_{index:02d}_pairs{stage['num_pairs']}_key{stage['key_size']}"
    metrics = stage_dir / 'metrics.jsonl'
    rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()] if metrics.exists() else []
    steps = [offset + row['step'] for row in rows]
    label = f"{stage['num_pairs']} pairs"
    axes[0].plot(steps, [row['validation']['fixed_length']['loss'] for row in rows], label=label)
    axes[1].plot(steps, [row['validation']['fixed_length']['exact_match'] for row in rows], label=label)
    offset += stage['step']
axes[0].set_ylabel('fixed-length validation CE (value + EOS)')
axes[1].set_ylabel('generated exact match (value + EOS)')
axes[1].axhline(.99, color='gray', linestyle='--')
axes[1].set_ylim(0, 1.03)
for ax in axes:
    ax.set_xlabel('total optimizer updates')
    ax.grid(alpha=.2)
    ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'convergence.png', dpi=150)
plt.show()
print('진행 확인: 각 stage의 metrics.txt')
print('공유: summary.json, evaluation.json, convergence.png 및 각 stage의 metrics.jsonl, resolved_config.yaml, metadata.json')
